# Speculatores v16 — Multi-Asset Pooled Optimizer

Run **top to bottom**. Optimizes both pivot sides on a *pool* of `(asset, timeframe)` streams (default: indices, 1D), **500 trials/side** (configurable), with persistent **resumable** storage on Drive.

**Before running:** put your TradingView daily exports in Google Drive at `MyDrive/cfd9/data/raw/`, named `{TICKER}_{TF}_*.csv` (e.g. `SPX_1D_*.csv`, `NDX_1D_*.csv`, `DAX_1D_*.csv`). The detector and Pine indicator are unchanged (parity preserved).

Flow: mount Drive → clone repo → set run settings → resolve pool → **run the diagnostic** (confirms the pool has enough structural events per fold) → launch (both sides) → provenance report.

The legacy **v15 single-asset** runner is preserved at the bottom (optional).


In [ ]:
# Cell 1 ? Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2 ? Clone or update repo and install deps
import os
from pathlib import Path
from datetime import datetime

REPO_DIR = Path('/content/cfd9')
REPO_URL = 'https://github.com/Sovenski/cfd9.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git rev-parse HEAD


In [ ]:
#@title v16 Run Settings  { display-mode: "form" }
#@markdown ## Speculatores v16 - run settings
#@markdown Fill the form, run this cell, then run the cells below in order.
#@markdown TradingView exports go in Google Drive at **`MyDrive/cfd9/data/raw_v16/`**
#@markdown (TV-native filenames are fine - no renaming).
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/cfd9')
DATA_DIR = DRIVE_ROOT / 'data' / 'raw_v16'
RESULTS_DIR = DRIVE_ROOT / 'results'
RUNS_DIR = DRIVE_ROOT / 'runs'
for _d in (DATA_DIR, RESULTS_DIR, RUNS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

#@markdown ---
#@markdown ### 1) Asset groups  (tick any combination to pool together)
USE_INDICES     = True   #@param {type:"boolean"}
USE_STOCKS      = False  #@param {type:"boolean"}
USE_COMMODITIES = True   #@param {type:"boolean"}
USE_WORLD_ETF   = True   #@param {type:"boolean"}
USE_FX          = True   #@param {type:"boolean"}
#@markdown - **INDICES** - SPX, NDX, DAX
#@markdown - **STOCKS** - 30 US large-caps (many events, but all one US-equity cluster)
#@markdown - **COMMODITIES** - gold / silver / platinum / palladium (oil only at 240/60)
#@markdown - **WORLD_ETF** - VT, VWCE
#@markdown - **FX** - 7 major pairs (EURUSD, GBPUSD, ...)
#@markdown Recommended default for daily: INDICES + COMMODITIES + WORLD_ETF + FX
#@markdown (~16 streams across 5 independent clusters).

#@markdown ---
#@markdown ### 2) Timeframe(s)  (tick one, or several to mix)
TF_1D  = True   #@param {type:"boolean"}
TF_240 = False  #@param {type:"boolean"}
TF_60  = False  #@param {type:"boolean"}
#@markdown - **1D** = daily (recommended; matches the macro-pivot goal)
#@markdown - **240 / 60** = 4h / 1h intraday (more independent assets, but mixing horizons
#@markdown   is a *scale-invariance bet* - the structural nest is in bars, so a 1h pivot is
#@markdown   an intraday swing, not a macro top). Validate before trusting a mixed run.

#@markdown ---
#@markdown ---
#@markdown ### 2b) History window (common era)
COMMON_ERA = "auto"  #@param ["auto", "all", "2000-01-01", "2010-01-01", "2015-01-01"]
#@markdown - **auto** (recommended) - concentrate folds where >=~half the streams
#@markdown   coexist, so every fold is multi-asset (avoids thin single-asset folds
#@markdown   from a long stream like SPX-1871). - **all** = use full history (old behaviour).
#@markdown   - a date = start folds at that date.
#@markdown ### 3) Volume policy
VOLUME_POLICY = "price_only"  #@param ["price_only", "volume_required", "mixed"]
#@markdown - **price_only** - ignore volume (robust default; raw index daily volume is patchy)
#@markdown - **volume_required** - use only each stream's real-volume date range (best for
#@markdown   LOW-side volume experiments); streams with no real volume are dropped
#@markdown - **mixed** - use volume where present, inert where missing

#@markdown ---
#@markdown ### 4) Optimizer budget
N_TRIALS = 500  #@param {type:"integer"}
SEED = 42       #@param {type:"integer"}
#@markdown ---
#@markdown ### 5) Sampler (search strategy)
SAMPLER = "CMAES"  #@param ["TPE", "CMAES", "BOTORCH"]
#@markdown - **CMAES** (recommended) - CMA-ES with integer margin; more sample-efficient
#@markdown   than TPE on this continuous-heavy space and cheap at high trial counts.
#@markdown   Categorical switches are routed to an inner TPE sampler.
#@markdown - **TPE** - the v16 / v16.1 default (multivariate, grouped). Natively handles
#@markdown   the mixed/categorical space and supports pruning.
#@markdown - **BOTORCH** - Gaussian-process Bayesian optimization (most sample-efficient
#@markdown   in low dimensions, but slower per trial and weaker with many categoricals
#@markdown   here). Installs `botorch` on first use.
#@markdown
#@markdown Note: CMAES / BOTORCH disable pruning (incomplete trials degrade their
#@markdown models); only TPE prunes.
#@markdown `N_TRIALS` is per side (HIGH and LOW each get this many). The run is resumable -
#@markdown if Colab disconnects, just re-run the launch cell and it continues.

# ---------------------------------------------------------------------------
# Assemble selections (do not edit below)
# ---------------------------------------------------------------------------
SELECTED_GROUPS = [g for g, on in {
    "INDICES": USE_INDICES, "STOCKS": USE_STOCKS, "COMMODITIES": USE_COMMODITIES,
    "WORLD_ETF": USE_WORLD_ETF, "FX": USE_FX,
}.items() if on]
SELECTED_TIMEFRAMES = [tf for tf, on in {
    "1D": TF_1D, "240": TF_240, "60": TF_60,
}.items() if on]
_ERA_KW = {} if COMMON_ERA == "auto" else ({"start": "1871-01-01"} if COMMON_ERA == "all" else {"start": COMMON_ERA})
RUN_SLUG = f"v16_{'-'.join(SELECTED_TIMEFRAMES) or 'none'}_{datetime.now():%Y%m%d_%H%M%S}"

assert SELECTED_GROUPS, "Tick at least one asset group (section 1)."
assert SELECTED_TIMEFRAMES, "Tick at least one timeframe (section 2)."
print("data dir   :", DATA_DIR)
print("CSV files  :", len(list(DATA_DIR.glob("*.csv"))), "present in", DATA_DIR.name)
print("groups     :", SELECTED_GROUPS)
print("timeframes :", SELECTED_TIMEFRAMES)
print("volume     :", VOLUME_POLICY, "| trials/side:", N_TRIALS, "| seed:", SEED, "| sampler:", SAMPLER)
print("run_slug   :", RUN_SLUG)


In [ ]:
# === v16 — resolve the pool ===
from src.universe import UNIVERSE, TIMEFRAMES, resolve_streams
print('available groups    :', list(UNIVERSE))
print('available timeframes :', TIMEFRAMES)
STREAMS = resolve_streams(SELECTED_GROUPS, SELECTED_TIMEFRAMES, data_dir=str(DATA_DIR))
print(f'resolved {len(STREAMS)} streams:', [s.stream_id for s in STREAMS])
if not STREAMS:
    print('\nWARNING: 0 streams resolved. Upload TV exports to', DATA_DIR,
          'as {TICKER}_{TF}_*.csv and re-run this cell.')


In [ ]:
#@title Step A - Pool sufficiency diagnostic (run BEFORE launch) { display-mode: "form" }
# === v16 — pre-run pool sufficiency diagnostic (run BEFORE launch) ===
from src.scoring import add_pivot_labels, REFERENCE_N
from src.pooled_validation import (
    StreamData, build_calendar_folds, load_stream_frame, apply_volume_policy,
)
import numpy as np
_TF_SECONDS = {"1D": 86400.0, "1W": 604800.0, "60": 3600.0, "240": 14400.0}

if not STREAMS:
    print('WARNING: 0 streams resolved — see the resolve cell above.')
else:
    _sd = []
    for s in STREAMS:
        df = load_stream_frame(s.path)
        df, keep = apply_volume_policy(df, policy=VOLUME_POLICY)
        if not keep:
            print(f'  drop {s.stream_id}: volume policy excluded it'); continue
        add_pivot_labels(df)
        _sd.append(StreamData(stream=s, df=df, bar_seconds=_TF_SECONDS[s.timeframe]))
    _folds = build_calendar_folds(_sd, **_ERA_KW)
    print(f'resolved streams = {[s.stream_id for s in STREAMS]}')
    print(f'calendar folds   = {len(_folds)}; streams/fold = {[len(f) for f in _folds]}')
    for side, lbl in (("HIGH", 1), ("LOW", -1)):
        per_fold = [sum(int((sl.df_oos[f'pivot_N{REFERENCE_N}'] == lbl).sum()) for sl in f)
                    for f in _folds]
        n_inf = sum(1 for x in per_fold if x > 0)
        print(f'  {side}: OOS pivots/fold = {per_fold} | informative = {n_inf}/{len(_folds)}'
              f' | mean = {np.mean(per_fold) if per_fold else 0:.1f}')
    print('\nRule of thumb: aim for >= ~5 informative folds, mean >= ~3 pivots/fold per '
          'side. If thin, add more index exports before launching.')


In [ ]:
#@title Step B - Launch optimization (both sides, resumable) { display-mode: "form" }
# === v16 — build pooled folds + launch BOTH sides (persistent, resumable) ===
import optuna
from src.scoring import add_pivot_labels
from src.monitor145 import make_storage
from src.pooled_validation import (
    StreamData, build_calendar_folds, build_pooled_optuna_objective,
    load_stream_frame, apply_volume_policy,
)
from src.speculatores145 import (
    params_from_trial, SEED_HEURISTIC_STRUCTURAL_HIGH, SEED_HEURISTIC_STRUCTURAL_LOW,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
if SAMPLER == "BOTORCH":
    import sys, subprocess
    try:
        import botorch  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "botorch", "optuna-integration"], check=True)
_TF_SECONDS = {"1D": 86400.0, "1W": 604800.0, "60": 3600.0, "240": 14400.0}

stream_datas = []
for s in STREAMS:
    df = load_stream_frame(s.path)
    df, keep = apply_volume_policy(df, policy=VOLUME_POLICY)
    if not keep:
        print(f'drop {s.stream_id}: volume policy excluded it'); continue
    add_pivot_labels(df)
    stream_datas.append(StreamData(stream=s, df=df, bar_seconds=_TF_SECONDS[s.timeframe]))

folds = build_calendar_folds(stream_datas, **_ERA_KW)
streams = [sd.stream for sd in stream_datas]
print(f'{len(folds)} folds; streams/fold = {[len(f) for f in folds]}')
print(f'sampler = {SAMPLER}')
assert folds, 'No folds — pool too small/short. Add more/longer exports (run the diagnostic).'

def _make_sampler():
    if SAMPLER == "CMAES":
        return optuna.samplers.CmaEsSampler(
            seed=SEED, with_margin=True,
            independent_sampler=optuna.samplers.TPESampler(
                multivariate=True, group=True, seed=SEED),
        )
    if SAMPLER == "BOTORCH":
        try:
            from optuna.integration import BoTorchSampler
        except ImportError:
            from optuna_integration import BoTorchSampler
        return BoTorchSampler(seed=SEED, n_startup_trials=20)
    return optuna.samplers.TPESampler(multivariate=True, group=True, seed=SEED)

def _make_pruner():
    # CMA-ES / BoTorch are degraded by pruning (incomplete evals disrupt the
    # covariance update / GP surrogate), so only TPE prunes.
    return optuna.pruners.MedianPruner() if SAMPLER == "TPE" else optuna.pruners.NopPruner()

_SEEDS = {"high": SEED_HEURISTIC_STRUCTURAL_HIGH, "low": SEED_HEURISTIC_STRUCTURAL_LOW}
for side in ("high", "low"):
    storage = make_storage(RUNS_DIR / f'{RUN_SLUG}_{side}.journal')
    study = optuna.create_study(
        study_name=f'spec_v16_{side}', direction='maximize',
        storage=storage, load_if_exists=True,
        sampler=_make_sampler(),
        pruner=_make_pruner(),
    )
    if not study.trials:
        study.enqueue_trial(_SEEDS[side])
    done = len([t for t in study.trials if t.state.is_finished()])
    remaining = max(0, N_TRIALS - done)
    print(f'[{side}] resuming at {done}/{N_TRIALS}; running {remaining} more ...')
    study.optimize(
        build_pooled_optuna_objective(folds, streams, params_from_trial, side),
        n_trials=remaining, show_progress_bar=True,
    )
    print(f'[{side}] best LCB = {study.best_value:.5f}')
    print(f'[{side}] best params = {study.best_params}')


In [ ]:
# === v16 — provenance report (reproducibility, spec 5.2) ===
import json, hashlib
from src.volume_quality import profile_volume

def _hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:12]

report = {
    'run_slug': RUN_SLUG,
    'selected_groups': SELECTED_GROUPS,
    'selected_timeframes': SELECTED_TIMEFRAMES,
    'volume_policy': VOLUME_POLICY,
    'n_trials_per_side': N_TRIALS,
    'sampler': SAMPLER,
    'n_folds': len(folds),
    'streams': [
        {'stream_id': sd.stream.stream_id, 'cluster': sd.stream.cluster_id,
         'bars': len(sd.df),
         'date_range': [str(sd.df.index[0]), str(sd.df.index[-1])],
         'data_hash': _hash(sd.stream.path),
         'volume_quality': profile_volume(sd.df).quality}
        for sd in stream_datas
    ],
}
out = RESULTS_DIR / f'{RUN_SLUG}_provenance.json'
out.write_text(json.dumps(report, indent=2))
print('wrote', out)
print(json.dumps(report, indent=2))


---
## Legacy: v15 single-asset runner (optional — NOT part of v16)

The cells below are the previous single-asset, script-backed pipeline (`run_speculatores_145.py`). They are kept for reference / one-off single-asset runs. **Do not run them as part of the v16 flow above** — they define their own config and launch a different optimizer.


In [ ]:
# Cell 3 - Run config
from pathlib import Path
from datetime import datetime
import re

DRIVE_ROOT = Path('/content/drive/MyDrive/cfd9')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_OPTIONS = {
    'SPX_1D': DRIVE_ROOT / 'data/raw/SPX_1D_18710201_20260318.csv',
    'DAX_1M': DRIVE_ROOT / 'data/raw/DAX_1M_20250113_20260227.csv',
}
DATASET_KEY = 'SPX_1D'
DATASET = str(DATASET_OPTIONS[DATASET_KEY])

TRIALS_PER_SIDE = 250
WORKERS_PER_SIDE = 4
STARTUP_TRIALS = 40
STABILITY_TRIALS = 50
SEED = 42
SKIP_CROSS_ASSET = False
RESUME_EXISTING = False

run_label = re.sub(r'[^a-z0-9]+', '_', DATASET_KEY.lower()).strip('_')
RUN_SLUG = f"{run_label}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
STUDY_PREFIX = f'speculatores_15_{RUN_SLUG}'

ACTIVE_ROOT = Path('/content/spec145_runs')
ACTIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = ACTIVE_ROOT / RUN_SLUG
RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR = DRIVE_ROOT / 'runs' / RUN_SLUG
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = RUN_DIR / 'spec145.journal'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / 'spec145.log'
PID_PATH = RUN_DIR / 'spec145.pid'

assert Path(DATASET).exists(), f'Missing dataset: {DATASET}'

print({
    'dataset_key': DATASET_KEY,
    'dataset': DATASET,
    'dataset_options': {k: str(v) for k, v in DATASET_OPTIONS.items()},
    'trials_per_side': TRIALS_PER_SIDE,
    'workers_per_side': WORKERS_PER_SIDE,
    'startup_trials': STARTUP_TRIALS,
    'stability_trials': STABILITY_TRIALS,
    'resume_existing': RESUME_EXISTING,
    'run_slug': RUN_SLUG,
    'run_dir': str(RUN_DIR),
    'drive_run_dir': str(DRIVE_RUN_DIR),
    'storage': str(STORAGE),
    'results_dir': str(RESULTS_DIR),
    'log_path': str(LOG_PATH),
    'pid_path': str(PID_PATH),
    'skip_cross_asset': SKIP_CROSS_ASSET,
})



In [ ]:
# Cell 4 ? Launch Speculatores 15 in background
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/cfd9').resolve()
assert REPO_DIR.exists(), REPO_DIR
assert (REPO_DIR / 'scripts' / 'run_speculatores_145.py').exists()

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'run_speculatores_145.py'),
    '--dataset', str(DATASET),
    '--trials-per-side', str(TRIALS_PER_SIDE),
    '--workers-per-side', str(WORKERS_PER_SIDE),
    '--startup-trials', str(STARTUP_TRIALS),
    '--stability-trials', str(STABILITY_TRIALS),
    '--seed', str(SEED),
    '--study-prefix', str(STUDY_PREFIX),
    '--storage', str(STORAGE),
    '--results-dir', str(RESULTS_DIR),
]
if SKIP_CROSS_ASSET:
    cmd.append('--skip-cross-asset')

if not RESUME_EXISTING and STORAGE.exists():
    STORAGE.unlink()

if PID_PATH.exists():
    try:
        old_pid = int(PID_PATH.read_text().strip())
        os.kill(old_pid, 0)
        raise RuntimeError(f'Run already active with PID {old_pid}. Stop it first or delete {PID_PATH}.')
    except OSError:
        PID_PATH.unlink(missing_ok=True)

log_handle = open(LOG_PATH, 'w', encoding='utf-8')
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)
PID_PATH.write_text(str(proc.pid), encoding='utf-8')
print('RUNNING:')
print(' '.join(cmd))
print(f'PID: {proc.pid}')
print(f'Log: {LOG_PATH}')


In [ ]:
# Cell 5 ? Monitor optimizer progress
import time
from pathlib import Path
from IPython.display import clear_output
from src.monitor145 import summarize_run, progress_dict


def _dataset_version_from_path(path: str) -> str:
    return Path(path).stem


def _bar(done: int, total: int, width: int = 24) -> str:
    total = max(total, 1)
    done = min(done, total)
    filled = int(width * done / total)
    return '[' + '#' * filled + '-' * (width - filled) + f'] {done}/{total}'


dataset_version = _dataset_version_from_path(DATASET)

for _ in range(10_000):
    clear_output(wait=True)
    pid_text = PID_PATH.read_text().strip() if PID_PATH.exists() else 'not running'
    print(f'PID: {pid_text}')
    print(f'Storage: {STORAGE}')
    print(f'Log: {LOG_PATH}')
    print('')
    try:
        summary = summarize_run(STORAGE, STUDY_PREFIX, dataset_version, TRIALS_PER_SIDE)
        total = progress_dict(summary)
        print('OPTIMIZER')
        print(_bar(total['total_done'], total['target']))
        print(f"running={total['running']} complete={total['completed']} pruned={total['pruned']} failed={total['failed']}")
        print('')
        for side, study in (('HIGH', summary.high), ('LOW', summary.low)):
            info = progress_dict(study)
            print(f"{side}: best={info['best_value']} p75={info['p75_value']}")
    except Exception as exc:
        print(f'Progress unavailable yet: {exc}')
        print('')

    if LOG_PATH.exists():
        print('LOG TAIL')
        lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-30:]))
        if any('Speculatores 15 report written to:' in line for line in lines[-30:]):
            PID_PATH.unlink(missing_ok=True)
            print('\nRun process finished.')
            break

    if PID_PATH.exists():
        try:
            import os
            os.kill(int(PID_PATH.read_text().strip()), 0)
        except OSError:
            PID_PATH.unlink(missing_ok=True)
            print('\nRun process finished.')
            break
    else:
        print('\nRun process finished.')
        break
    time.sleep(10)


In [ ]:
# Cell 6 ? Show this run's report and preview
from pathlib import Path

reports = sorted(Path(RESULTS_DIR).glob(f'*{Path(DATASET).stem}*__speculatores_15_pathA.md'), key=lambda p: p.stat().st_mtime, reverse=True)
assert reports, 'No reports found.'

current_run_reports = [p for p in reports if RUN_SLUG in p.name]
report_path = current_run_reports[0] if current_run_reports else reports[0]
print(f'Report: {report_path}')
print(report_path.read_text(encoding='utf-8'))


In [ ]:
# Cell 7 ? Optional: inspect parity section only
text = report_path.read_text(encoding='utf-8')
marker = '## Cell 3.3'
idx = text.find(marker)
if idx >= 0:
    print(text[idx:idx+2500])
else:
    print('Parity section not found.')
